# Модуль 06 · Урок 1. Векторизация, скорость и профилирование

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Почему циклы часто проигрывают
В pandas столбец хранится как массив данных, поэтому арифметика по Series выполняется без Python-цикла по каждому элементу. Например, `df['qty']*df['price']` обычно гораздо быстрее, чем `for _,row in df.iterrows(): ...`. `apply(axis=1)` удобен, но зачастую вызывает Python-функцию для каждой строки и не является «автоматической оптимизацией». Важно сначала написать правильное решение, затем измерить скорость на типичном объёме.

In [ ]:
import pandas as pd
prices = pd.DataFrame({'p':[5,10,20,15], 'q':[2,4,1,3]})
prices['total'] = prices.p*prices.q
print(prices)

## 2. Измерение времени
В VS Code можно использовать магические команды ноутбука `%timeit expression` и `%%timeit` для измерения повторных запусков. В обычном Python-коде используйте `time.perf_counter` и несколько прогонов. Первый запуск может включать прогрев кэшей, чтение файла или выделение памяти. Сравнивайте методы на **одних и тех же данных**, проверив равенство ответов. Не оптимизируйте по интуиции.

In [ ]:
import time
start = time.perf_counter()
result = (prices.p*prices.q).sum()
print('result',result,'seconds',time.perf_counter()-start)

## 3. Использование map, where, groupby вместо apply
Для кодирования категорий подходит `map`, для двух условий — `where`/`np.where`, для работы внутри групп — `groupby.transform`. Векторные строковые операции `.str` и временные `.dt` нередко заменяют циклы. Но не любая векторизация обязательно быстрее: большие промежуточные массивы увеличивают использование памяти. Оптимизация всегда баланс между временем, RAM и читаемостью.

## 4. Профилирование и критерии качества
Используйте `df.memory_usage(deep=True)` для базовой оценки, `%timeit` для времени и отдельную проверку результата на крайних случаях. Если код стал быстрее на 5%, но потерял понятность и качество, преимущество сомнительно. Запишите в отчёте исходные условия: версия pandas, размер датасета, процессор, число повторов, расход памяти.

**Проверьте себя:** почему `apply(axis=1)` может быть медленным? Чем профилирование отличается от оптимизации? Почему сравнение времени без сравнения результата недостоверно?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m6_01 · vector total

Без явных циклов получите Series `total` = price*qty*(1-discount/100), с индексом исходной таблицы.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_01: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m6_01 = expected_for("m6_01", df=df)


In [ ]:
# m6_01: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 6 значений; имя серии: `total`.

```text
z    254.0
b    222.3
a    368.6
d    317.0
x    136.8
q      0.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_01", result, _expected_m6_01)

### m6_02 · vector flag

Получите логическую Series `flag`: заказ оплачен, а его стоимость без скидки >= threshold.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `threshold` — входной параметр (значение видно в примере); в этом примере: `180`.


**Выполнение:** используйте готовые переменные `df`, `threshold`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_02: используются переменные df, threshold

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

threshold = 180

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m6_02 = expected_for("m6_02", df=df, threshold=threshold)


In [ ]:
# m6_02: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, threshold
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 6 значений; имя серии: `flag`.

```text
z     True
b    False
a     True
d     True
x    False
q    False
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_02", result, _expected_m6_02)

### m6_03 · vector norm

Получите Series `normalized`: отношение qty к максимальному qty в своём городе. Заказы с нулевыми максимумами получают NA.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_03: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m6_03 = expected_for("m6_03", df=df)


In [ ]:
# m6_03: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 6 значений; имя серии: `normalized`.

```text
z    0.500000
b    0.333333
a    1.000000
d    1.000000
x    1.000000
q    0.000000
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_03", result, _expected_m6_03)

### m6_04 · compact status

Получите новый DataFrame со всеми колонками, где status представлен в типе category.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_04: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m6_04 = expected_for("m6_04", df=df)


In [ ]:
# m6_04: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 10 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment
z  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно
d  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок
x  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                 
q  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_04", result, _expected_m6_04)

### m6_05 · bulk replace

Добавьте `is_returned` (bool): True только у cancelled; `status_code` (целое): paid→2,new→1,cancelled→0. Используйте векторную обработку.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_05: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m6_05 = expected_for("m6_05", df=df)


In [ ]:
# m6_05: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 12 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`, `is_returned`, `status_code`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment  is_returned  status_code
z  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...        False            2
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка        False            1
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно        False            2
d  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок        False            2
x  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                          True            0
q  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None        False            1
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_05", result, _expected_m6_05)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()